# Run a trained ACT policy in the MuJoCo simulator

Start a MuJoCo simulation, download a trained **ACT** policy from the Hugging Face Hub and let it work in a simulated **conveyor sorting cell**,
from one dashboard in this notebook. No robot and no GPU needed.

Items ride a belt past an SO-101 arm. The arm has to pick each one and drop it into the right bin:
**red, blue and green items go into the bin of their colour, purple or cracked items into the reject bin.**

The policy was trained with [Physical AI Studio](https://github.com/open-edge-platform/physical-ai-studio) on about 100 episodes recorded
from the simulation's built-in demonstrator, then exported to OpenVINO.

## Setup

Until the packages are published, run this notebook from a checkout of the repository, with its own environment:

```bash
uv sync --all-packages --extra notebooks
uv run jupyter lab packages/physicalai-mujoco-plugin/examples
```

(Once `physicalai` and the plugin are on PyPI, `pip install "physicalai[plugin-mujoco,transport,notebooks]"` is all you need.)

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
import threading
import time
import urllib.request

import ipywidgets as widgets
from IPython.display import display

# ACT policy, OpenVINO export. A Hub id, or a local folder with `manifest.json` via ACT_MODEL_DIR.
MODEL_REPO = os.environ.get("ACT_MODEL_DIR") or "iMaxxgx/act-so101-conveyor-sort"
SIM_NAME = "notebook-sim"  # name the simulation registers under

## 1. Start the simulation

`physicalai-mujoco start` runs the simulation in its own process. With `--status-json` it reports its progress as JSON lines and ends with a `ready` event
that lists the ports it picked (`--http-port 0` and `--viser-port 0` mean "any free port"). With `--exit-with-parent` it stops when this
notebook's pipe to it closes, so it cannot be left running by accident.

In [ ]:
sim_log = tempfile.NamedTemporaryFile("w+", prefix="mujoco-sim-", suffix=".log", delete=False)  # the simulation's own log
sim = subprocess.Popen(
    [
        sys.executable, "-m", "physicalai_mujoco_plugin", "start",
        "--profile", "so101", "--scene", "conveyor_sort", "--name", SIM_NAME,
        "--status-json", "--exit-with-parent", "--http-port", "0", "--viser-port", "0",
    ],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=sim_log, text=True,
)

ready = None
for line in sim.stdout:
    event = json.loads(line)
    print(event)
    if event["event"] == "ready":
        ready = event
        break
    if event["event"] == "error":
        raise RuntimeError(f"{event['message']} (log: {sim_log.name})")

http_url, viewer_url = ready["http_url"], ready["viewer_url"]

## 2. Load the policy and build the runtime

`InferenceModel.from_pretrained` downloads the policy package (an OpenVINO model plus a `manifest.json` that says which inputs it needs and how to preprocess them) and compiles it.

A `RobotRuntime` then ties four things together, as on hardware:

- the **robot**: `SharedRobot` attaches to the simulation by its name,
- the **cameras**: the simulation's two MJPEG streams, served like a real robot's IP cameras. The names (`gripper`, `overview`) must match the policy's inputs,
- the **action source**: `PolicySource` runs the policy. ACT predicts a chunk of 100 actions; `request_threshold=0.98` asks for a fresh chunk after about every second action, which this policy needs
  (it does not recover from a missed grasp by itself),
- the **loop rate**: 30 Hz.

In [ ]:
from physicalai.capture import IPCamera
from physicalai.inference import InferenceModel
from physicalai.robot import SharedRobot
from physicalai.runtime import ChunkedActionQueue, LerpSmoother, PolicySource, RobotRuntime, SyncExecution

policy = InferenceModel.from_pretrained(MODEL_REPO, backend="openvino", device="CPU")

runtime = RobotRuntime(
    robot=SharedRobot(name=SIM_NAME),
    action_source=PolicySource(
        model=policy,
        execution=SyncExecution(request_threshold=0.98),
        action_queue=ChunkedActionQueue(smoother=LerpSmoother()),
    ),
    fps=30.0,
    cameras={
        "overview": IPCamera(url=f"{http_url}/cameras/overview/mjpeg", width=640, height=480, fps=30),
        "gripper": IPCamera(url=f"{http_url}/cameras/wrist/mjpeg", width=640, height=480, fps=30),
    },
)
runtime.connect()  # attaches to the simulation and the cameras; stays connected between runs

## 3. The dashboard

Everything in one place: the simulation's 3D viewer (drag to orbit, scroll to zoom in, drag items around), the two cameras the policy sees, the score, and the controls.

- **Run policy** / **Stop**: start and stop the policy. The arm stays where it is when you stop.
- **Reset scene**: respawn the items and start a new episode.
- **Belt speed**: how fast items arrive. The policy was trained at 3 cm/s, so expect more misses when you speed it up.
- **Shut down**: stop the policy and the simulation when you are done.

The score counts items that ended up in the bin they belong in (**correct**), in a wrong bin (**wrong**), or off the belt (**missed**).

> Running Jupyter on another machine? The viewer URL points at *that* machine's `localhost`. Forward the port, or start the simulation with `--viser-host 0.0.0.0`.

In [ ]:
def request(path: str, body: dict | None = None) -> bytes:
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(f"{http_url}{path}", data=data, headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as response:
        return response.read()


# --- widgets -----------------------------------------------------------------------------------------------
viewer = widgets.HTML(f'<iframe src="{viewer_url}" style="width:100%;height:600px;border:1px solid #ccc"></iframe>')
cameras = {name: widgets.Image(format="jpeg", width=360) for name in ("overview", "wrist")}
status = widgets.HTML()
run_button = widgets.Button(description="Run policy", icon="play", button_style="success")
stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger", disabled=True)
reset_button = widgets.Button(description="Reset scene", icon="refresh")
shutdown_button = widgets.Button(description="Shut down", icon="power-off", tooltip="Stop the policy and the simulation")
speed = widgets.FloatSlider(
    value=3.0, min=0.0, max=10.0, step=0.5, description="Belt cm/s", readout_format=".1f", continuous_update=False
)

# --- behaviour ---------------------------------------------------------------------------------------------
run = {"thread": None, "error": None, "started": 0.0}


def _run_policy() -> None:
    try:
        runtime.run()  # until stop() is called
    except Exception as exc:  # noqa: BLE001 - show any failure in the dashboard
        run["error"] = repr(exc)
    finally:
        run_button.disabled, stop_button.disabled = False, True


def on_run(_: widgets.Button) -> None:
    run_button.disabled, stop_button.disabled = True, False
    run["error"], run["started"] = None, time.monotonic()
    run["thread"] = threading.Thread(target=_run_policy, daemon=True)
    run["thread"].start()


def on_stop(_: widgets.Button) -> None:
    stop_button.disabled = True
    runtime.stop()


def on_shutdown(_: widgets.Button) -> None:
    """Stop the refresh thread, the policy and the simulation (closing its stdin stops it)."""
    ui_stop.set()
    runtime.stop()
    time.sleep(0.5)  # let the control loop finish its last tick
    runtime.disconnect()
    sim.stdin.close()
    sim.wait(timeout=30)
    for button in (run_button, stop_button, reset_button, shutdown_button):
        button.disabled = True
    speed.disabled = True
    status.value = "<b>simulation stopped</b>"


run_button.on_click(on_run)
stop_button.on_click(on_stop)
reset_button.on_click(lambda _: request("/reset", {}))
shutdown_button.on_click(on_shutdown)
speed.observe(lambda change: request("/conveyor/belt-speed", {"speed": change["new"] / 100}), names="value")


def render_status() -> str:
    episode = json.loads(request("/health"))["episode"]
    score = episode["score"]
    running = run["thread"] is not None and run["thread"].is_alive()
    state = f"running {time.monotonic() - run['started']:.0f} s" if running else "idle"
    if run["error"]:
        state = f"<span style='color:#b00'>stopped: {run['error']}</span>"
    return (
        f"<b>{state}</b> &nbsp;|&nbsp; items fed {episode['spawned']} of {episode['items_per_episode']}"
        f" &nbsp;|&nbsp; <span style='color:#080'>correct {score['correct']}</span>"
        f" &nbsp; <span style='color:#b00'>wrong {score['wrong']}</span>"
        f" &nbsp; <span style='color:#a60'>missed {score['missed']}</span>"
        f" &nbsp;|&nbsp; episodes finished {episode['episode_count']}"
    )


ui_stop = threading.Event()


def _refresh() -> None:
    """Stream the cameras at about 10 frames per second and refresh the score twice a second."""
    tick = 0
    while not ui_stop.is_set():
        try:
            for name, image in cameras.items():
                image.value = request(f"/cameras/{name}/frame.jpg")
            if tick % 5 == 0:
                status.value = render_status()
        except OSError:
            pass  # the simulation is not answering (yet, or any more)
        tick += 1
        ui_stop.wait(0.1)


threading.Thread(target=_refresh, daemon=True).start()

# --- layout ------------------------------------------------------------------------------------------------
controls = widgets.HBox([run_button, stop_button, reset_button, speed, shutdown_button])
side = widgets.VBox([widgets.HTML("<b>overview</b>"), cameras["overview"], widgets.HTML("<b>gripper</b>"), cameras["wrist"]])
panels = widgets.GridBox(
    [viewer, side], layout=widgets.Layout(grid_template_columns="minmax(480px, 1fr) 380px", grid_gap="12px", width="100%")
)
dashboard = widgets.VBox([controls, status, panels])
display(dashboard)

## Where next

- Start another robot or scene: `physicalai-mujoco profiles` lists the robots, and the [plugin README](../README.md) describes the scenes.
- Record your own demonstrations, train in Physical AI Studio and point `MODEL_REPO` at your own export.
- Run the same policy from the command line: `physicalai run --config runtime.yaml`.